# refshift -- consolidated mismatch experiments

CSP+LDA, ShallowConvNet, EEGNet, ATCNet mismatch matrices (no-EA and EA),
full jitter, leave-one-reference-out (LORO), and leave-one-family-out (LOFO),
with operator-family analysis. Set `DATASET` in section B and run top to bottom.

## A -- Setup (run once per kernel)

### A1. Install refshift

The lean package ships as a Kaggle dataset. Point `REFSHIFT_SRC` at the folder
that contains `pyproject.toml` (adjust the path to match your attached
dataset), then install it editable. We also pin mne for braindecode.

In [ ]:
import os, subprocess, sys, importlib

# Folder containing pyproject.toml inside your attached Kaggle dataset.
REFSHIFT_SRC = "/kaggle/input/datasets/delhialli/refshift-lean/refshift-lean"
assert os.path.isfile(os.path.join(REFSHIFT_SRC, "pyproject.toml")), \
    f"pyproject.toml not found under {REFSHIFT_SRC}; fix REFSHIFT_SRC"

subprocess.run(["pip", "install", "-q", "-e", f"{REFSHIFT_SRC}[dl]"], check=True)
subprocess.run(["pip", "install", "-q", "moabb", "braindecode"], check=True)
subprocess.run(["pip", "uninstall", "-y", "-q", "mne", "mne-bids"], check=False)
subprocess.run(
    ["pip", "install", "-q", "--no-cache-dir", "mne==1.11.0", "mne-bids>=0.18"],
    check=True,
)

# An editable install adds a .pth read at startup; a kernel that began before
# the install won't see it. Add the src dir to sys.path so this kernel can import.
if REFSHIFT_SRC not in sys.path:
    sys.path.insert(0, REFSHIFT_SRC)
importlib.invalidate_caches()
print("install done")

### A2. Environment setup + Kaggle dataset symlinks

`setup_kaggle_env()` sets MNE_DATA / thread caps and symlinks your attached
Kaggle datasets into MOABB's expected cache layout, so nothing downloads. Run
this BEFORE any data load.

In [ ]:
from refshift import setup_kaggle_env
setup_kaggle_env()   # symlinks all five datasets; idempotent

### A3. Imports, paths, helpers

In [ ]:
import time, warnings
import numpy as np
import pandas as pd

from refshift import (
    calibrate_csp_lda,
    run_mismatch, run_mismatch_jitter, run_loro_matrix, run_lofo_matrix,
    mismatch_matrix, mismatch_std_matrix,
    REFERENCE_MODES, FAMILIES, reference_modes_for_dataset, canonical_mode_tuple,
    report_matrix, report_families, report_jitter_full, report_loro, report_lofo,
)

import mne
mne.set_log_level("WARNING")
warnings.filterwarnings("ignore", message=".*pick_types.*legacy.*")
os.environ["MNE_LOGGING_LEVEL"] = "ERROR"
import mne.filter
_orig_filter = mne.filter.filter_data
def _quiet_filter(*args, **kwargs):
    kwargs["verbose"] = "ERROR"
    return _orig_filter(*args, **kwargs)
mne.filter.filter_data = _quiet_filter

BASE    = "/kaggle/working"
CACHE   = f"{BASE}/cache"
RESULTS = f"{BASE}/results"
FIGS    = f"{BASE}/figs"
for d in (CACHE, RESULTS, FIGS):
    os.makedirs(d, exist_ok=True)


def _run_or_skip(name, fn, force=False):
    """Run fn() -> DataFrame, cache to CSV, or reload an existing CSV."""
    path = f"{RESULTS}/{name}.csv"
    if (not force) and os.path.exists(path):
        df = pd.read_csv(path)
        print(f"[SKIP] {name} (loaded {len(df)} rows)")
        return df
    t0 = time.time()
    df = fn()
    df.to_csv(path, index=False)
    print(f"[DONE] {name}  rows={len(df)}  took {(time.time()-t0)/60:.1f} min")
    return df

print("paths ready:", RESULTS)

## B -- Config (the only cell you edit)

In [ ]:
DATASET    = "iv2a"        # iv2a | openbmi | cho2017 | dreyer2023 | schirrmeister2017

# None -> the dataset-safe default set (drops cz_ref for schirrmeister).
# Or pass an explicit set, e.g. {"native", "car", "median", "rest", "cz_ref",
# "lap_small", "lap_large"}.
REFERENCES = None
SEEDS      = [0, 1, 2]     # DL runs average over these; CSP+LDA uses seed [0]
TAG        = "consolidated"
FORCE      = False         # True = ignore cached CSVs and recompute

DL_MAX_EPOCHS = 200
DL_BATCH_SIZE = 32

MODES = (reference_modes_for_dataset(DATASET) if REFERENCES is None
         else canonical_mode_tuple(REFERENCES))
print(f"DATASET = {DATASET}")
print(f"MODES   = {MODES}  ({len(MODES)} operators)")
print(f"SEEDS   = {SEEDS}")

## C -- Calibration (sanity check, IV-2a only)

Confirms bare CSP+LDA reproduces the MOABB baseline and that a no-op reference
transformer changes nothing. Skipped for non-IV-2a datasets.

In [ ]:
if DATASET == "iv2a":
    results, summary, passed = calibrate_csp_lda("iv2a", subjects=[1])
    print("passed:", passed)
else:
    print(f"calibration skipped for {DATASET}")

## D -- Mismatch matrices, no EA

### D1. CSP+LDA (no EA)

In [ ]:
NAME = f"{DATASET}_csp_lda_mismatch_noEA_{TAG}"
df_csp = _run_or_skip(NAME, lambda: run_mismatch(
    DATASET, model="csp_lda", seeds=[0],
    reference_modes=REFERENCES, apply_ea=False,
), force=FORCE)
report_matrix(df_csp, title=f"CSP+LDA (no EA) -- {DATASET}", modes=MODES)
report_families(df_csp, title=f"CSP+LDA (no EA) -- {DATASET}", modes=MODES)

### D2. ShallowConvNet (no EA)

In [ ]:
NAME = f"{DATASET}_shallow_mismatch_noEA_{TAG}"
df_shallow = _run_or_skip(NAME, lambda: run_mismatch(
    DATASET, model="shallow", seeds=SEEDS,
    reference_modes=REFERENCES, apply_ea=False,
    dl_max_epochs=DL_MAX_EPOCHS, dl_batch_size=DL_BATCH_SIZE, cache_dir=CACHE,
), force=FORCE)
report_matrix(df_shallow, title=f"ShallowConvNet (no EA) -- {DATASET}", modes=MODES)
report_families(df_shallow, title=f"ShallowConvNet (no EA) -- {DATASET}", modes=MODES)

### D3. EEGNet (no EA)

In [ ]:
NAME = f"{DATASET}_eegnet_mismatch_noEA_{TAG}"
df_eegnet = _run_or_skip(NAME, lambda: run_mismatch(
    DATASET, model="eegnet", seeds=SEEDS,
    reference_modes=REFERENCES, apply_ea=False,
    dl_max_epochs=DL_MAX_EPOCHS, dl_batch_size=DL_BATCH_SIZE, cache_dir=CACHE,
), force=FORCE)
report_matrix(df_eegnet, title=f"EEGNet (no EA) -- {DATASET}", modes=MODES)
report_families(df_eegnet, title=f"EEGNet (no EA) -- {DATASET}", modes=MODES)

### D4. ATCNet (no EA)

In [ ]:
NAME = f"{DATASET}_atcnet_mismatch_noEA_{TAG}"
df_atcnet = _run_or_skip(NAME, lambda: run_mismatch(
    DATASET, model="atcnet", seeds=SEEDS,
    reference_modes=REFERENCES, apply_ea=False,
    dl_max_epochs=DL_MAX_EPOCHS, dl_batch_size=DL_BATCH_SIZE, cache_dir=CACHE,
), force=FORCE)
report_matrix(df_atcnet, title=f"ATCNet (no EA) -- {DATASET}", modes=MODES)
report_families(df_atcnet, title=f"ATCNet (no EA) -- {DATASET}", modes=MODES)

## E -- Mismatch matrices, with EA

### E1. CSP+LDA (EA)

In [ ]:
NAME = f"{DATASET}_csp_lda_mismatch_EA_{TAG}"
df_csp_ea = _run_or_skip(NAME, lambda: run_mismatch(
    DATASET, model="csp_lda", seeds=[0],
    reference_modes=REFERENCES, apply_ea=True,
), force=FORCE)
report_matrix(df_csp_ea, title=f"CSP+LDA (EA) -- {DATASET}", modes=MODES)
report_families(df_csp_ea, title=f"CSP+LDA (EA) -- {DATASET}", modes=MODES)

### E2. ShallowConvNet (EA)

In [ ]:
NAME = f"{DATASET}_shallow_mismatch_EA_{TAG}"
df_shallow_ea = _run_or_skip(NAME, lambda: run_mismatch(
    DATASET, model="shallow", seeds=SEEDS,
    reference_modes=REFERENCES, apply_ea=True,
    dl_max_epochs=DL_MAX_EPOCHS, dl_batch_size=DL_BATCH_SIZE, cache_dir=CACHE,
), force=FORCE)
report_matrix(df_shallow_ea, title=f"ShallowConvNet (EA) -- {DATASET}", modes=MODES)
report_families(df_shallow_ea, title=f"ShallowConvNet (EA) -- {DATASET}", modes=MODES)

## G -- Full jitter (DL-only)

### G1. Full jitter -- ShallowConvNet

In [ ]:
NAME = f"{DATASET}_shallow_jitter_full_{TAG}"
df_shallow_jit = _run_or_skip(NAME, lambda: run_mismatch_jitter(
    DATASET, model="shallow", condition="full", seeds=SEEDS,
    reference_modes=REFERENCES,
    dl_max_epochs=DL_MAX_EPOCHS, dl_batch_size=DL_BATCH_SIZE, cache_dir=CACHE,
), force=FORCE)
report_jitter_full(df_shallow_jit, title=f"Full jitter -- ShallowConvNet -- {DATASET}", modes=MODES)

### G2. Full jitter -- EEGNet

In [ ]:
NAME = f"{DATASET}_eegnet_jitter_full_{TAG}"
df_eegnet_jit = _run_or_skip(NAME, lambda: run_mismatch_jitter(
    DATASET, model="eegnet", condition="full", seeds=SEEDS,
    reference_modes=REFERENCES,
    dl_max_epochs=DL_MAX_EPOCHS, dl_batch_size=DL_BATCH_SIZE, cache_dir=CACHE,
), force=FORCE)
report_jitter_full(df_eegnet_jit, title=f"Full jitter -- EEGNet -- {DATASET}", modes=MODES)

### G3. Full jitter -- ATCNet

In [ ]:
NAME = f"{DATASET}_atcnet_jitter_full_{TAG}"
df_atcnet_jit = _run_or_skip(NAME, lambda: run_mismatch_jitter(
    DATASET, model="atcnet", condition="full", seeds=SEEDS,
    reference_modes=REFERENCES,
    dl_max_epochs=DL_MAX_EPOCHS, dl_batch_size=DL_BATCH_SIZE, cache_dir=CACHE,
), force=FORCE)
report_jitter_full(df_atcnet_jit, title=f"Full jitter -- ATCNet -- {DATASET}", modes=MODES)

## H -- Leave-One-Reference-Out (LORO)

Hold out one reference at a time: train jitter over the rest, test transfer to
the held-out reference. The diagonal of the matrix is the unseen-reference
accuracy; the recovery gap is the cost of never training on a reference.

### H1. LORO -- ShallowConvNet

In [ ]:
NAME = f"{DATASET}_shallow_loro_{TAG}"
df_shallow_loro = _run_or_skip(NAME, lambda: run_loro_matrix(
    DATASET, model="shallow", seeds=SEEDS,
    reference_modes=REFERENCES,
    dl_max_epochs=DL_MAX_EPOCHS, dl_batch_size=DL_BATCH_SIZE, cache_dir=CACHE,
), force=FORCE)
report_loro(df_shallow_loro, title=f"LORO -- ShallowConvNet -- {DATASET}", modes=MODES)

## I -- Leave-One-Family-Out (LOFO)

Hold out a whole family of references (global / single / spatial), train jitter
over the others, test on every reference. The family x family matrix shows
whether the model generalises across kinds of reference operation.

### I1. LOFO -- ShallowConvNet

In [ ]:
NAME = f"{DATASET}_shallow_lofo_{TAG}"
df_shallow_lofo = _run_or_skip(NAME, lambda: run_lofo_matrix(
    DATASET, model="shallow", families=FAMILIES, seeds=SEEDS,
    reference_modes=REFERENCES,
    dl_max_epochs=DL_MAX_EPOCHS, dl_batch_size=DL_BATCH_SIZE, cache_dir=CACHE,
), force=FORCE)
report_lofo(df_shallow_lofo, title=f"LOFO -- ShallowConvNet -- {DATASET}")